# Flask – Practice Exercises

Hands-on exercises for Flask, following `g_fw-flask.ipynb`.

Use `app.test_client()` to send requests without starting a server.

### Contents

- [Exercise 1 – Hello World Route](#exercise-1--hello-world-route)
- [Exercise 2 – URL Path and Query Parameters](#exercise-2--url-path-and-query-parameters)
- [Exercise 3 – JSON Endpoints and the Request Object](#exercise-3--json-endpoints-and-the-request-object)
- [Exercise 4 – Data Validation with Pydantic](#exercise-4--data-validation-with-pydantic)
- [Exercise 5 – In-Memory Trading Endpoints](#exercise-5--in-memory-trading-endpoints)
- [Exercise 6 – Blueprints and Application Factory](#exercise-6--blueprints-and-application-factory)
- [Exercise 7 – Error Handling and Custom Responses](#exercise-7--error-handling-and-custom-responses)
- [Exercise 8 – Testing Flask Apps](#exercise-8--testing-flask-apps)
- [Exercise 9 – Background Work and Async Considerations](#exercise-9--background-work-and-async-considerations)
- [Exercise 10 – Project Structure](#exercise-10--project-structure)

## Exercise 1 – Hello World Route

**Goal**: Create a minimal Flask application.

**Requirements**:

- Instantiate `app = Flask(__name__)` and add `GET /` returning a plain text greeting.
- Add `GET /ping` returning JSON `{"status": "ok"}`.
- Use `app.test_client()` to call both routes and assert status 200.

**Hint**: Set `app.config['TESTING'] = True` before using the test client.

## Exercise 2 – URL Path and Query Parameters

**Goal**: Accept parameters from the URL.

**Requirements**:

- Add `GET /symbols/<ticker>` that returns the ticker uppercased as JSON.
- Add a `precision: int = 2` query parameter that controls decimal rounding on a price.
- Test with a missing `ticker` (404) and a valid one (200).

**Hint**: Use `request.args.get('key', default)` for query parameters.

## Exercise 3 – JSON Endpoints and the Request Object

**Goal**: Read and return JSON bodies.

**Requirements**:

- Add `POST /orders` that reads `request.get_json()` and returns the body with an added `id` field.
- Return 400 with a message if the body is missing or malformed.
- Test with `client.post('/orders', json={...})`.

**Hint**: `request.get_json(silent=True)` returns `None` instead of raising on bad JSON.

## Exercise 4 – Data Validation with Pydantic

**Goal**: Use Pydantic to validate request data inside a Flask route.

**Requirements**:

- Define `OrderIn(BaseModel)` with `symbol`, `qty: int`, `price: float`.
- In the POST `/orders` route, call `OrderIn(**request.get_json())` inside a try/except `ValidationError`.
- Return 422 with the validation errors on failure.

**Hint**: Pydantic `ValidationError.errors()` returns a list of dicts with `loc`, `msg`, and `type`.

## Exercise 5 – In-Memory Trading Endpoints

**Goal**: Build a small stateful API backed by a Python dict.

**Requirements**:

- Maintain `ORDERS: dict[int, dict]` at module level.
- Implement `GET /orders`, `POST /orders`, `GET /orders/<id>`, `DELETE /orders/<id>`.
- Return 404 for unknown ids; return the list sorted by id.

**Hint**: `abort(404)` raises a `404 Not Found` response from anywhere in a route.

## Exercise 6 – Blueprints and Application Factory

**Goal**: Split routes across Blueprints and use the app factory pattern.

**Requirements**:

- Create a `market_bp = Blueprint('market', __name__, url_prefix='/market')` with one route.
- Create `orders_bp` with another route.
- Write `create_app(config=None)` that registers both blueprints and returns the app.

**Hint**: The factory pattern lets you create multiple app instances with different configs.

## Exercise 7 – Error Handling and Custom Responses

**Goal**: Return consistent error shapes.

**Requirements**:

- Register `@app.errorhandler(404)` and `@app.errorhandler(500)` returning JSON `{"error": ...}`.
- Raise `abort(404)` inside a route and confirm the handler fires.
- Write a helper `json_error(code, message)` reused by all handlers.

**Hint**: `@app.errorhandler(Exception)` catches all unhandled exceptions.

## Exercise 8 – Testing Flask Apps

**Goal**: Write a complete test suite using the test client.

**Requirements**:

- Write `TestOrdersAPI(unittest.TestCase)` using `setUp` to create a fresh app and client.
- Test create, list, get, and delete operations.
- Assert HTTP status codes, JSON keys, and state changes.

**Hint**: Use the app factory from Exercise 6 with a `TESTING=True` config for isolation.

## Exercise 9 – Background Work and Async Considerations

**Goal**: Understand Flask's synchronous nature and workarounds.

**Requirements**:

- Add a `POST /reports/generate` endpoint that simulates slow work with `time.sleep(0.1)`.
- Run it with `threading.Thread(target=generate_report, daemon=True).start()` to return immediately.
- Explain why a real deployment would use Celery or RQ instead.

**Hint**: Flask itself is synchronous by default; `async def` routes require an ASGI server.

## Exercise 10 – Project Structure

**Goal**: Organise a production-ready Flask project.

**Requirements**:

- Print the recommended layout: `app/__init__.py` (factory), `app/routes/`, `app/models.py`, `config.py`.
- Show how `config.py` defines `DevelopmentConfig` and `ProductionConfig` classes.
- List three Flask-specific best practices.

**Hint**: Use `flask run` with `FLASK_ENV=development` for local dev; never expose the dev server publicly.